<a href="https://colab.research.google.com/github/ajsarsva/video-captioning-thesis/blob/main/notebooks/26_gradio_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

if os.path.exists('/content/video-captioning-thesis'):
    %cd /content/video-captioning-thesis
    !git pull origin main
else:
    !git clone https://github.com/ajsarsva/video-captioning-thesis.git
    %cd /content/video-captioning-thesis

import sys
sys.path.append('/content/video-captioning-thesis/src')

print("✅ Ready!")

Mounted at /content/drive
Cloning into 'video-captioning-thesis'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 121 (delta 66), reused 27 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 20.02 MiB | 8.85 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/video-captioning-thesis
✅ Ready!


Install Dependencies

In [2]:
!pip install gradio yt-dlp transformers accelerate scikit-image sentencepiece -q

import os, sys, time, cv2
import numpy as np
import torch
from PIL import Image
import gradio as gr

sys.path.append('/content/video-captioning-thesis/src')

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("✅ Dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.5 MB/s eta 0:00:00
GPU: Tesla T4
✅ Dependencies installed!


Load All Models

In [3]:
from transformers import (BlipProcessor, BlipForConditionalGeneration,
                          CLIPProcessor, CLIPModel,
                          AutoTokenizer, AutoModelForSeq2SeqLM)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading BLIP...")
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base").to(device)
print("✅ BLIP ready!")

print("Loading CLIP...")
clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32").to(device)
print("✅ CLIP ready!")

print("Loading FLAN-T5...")
t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
t5_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base").to(device)
print("✅ FLAN-T5 ready!")

print(f"\nAll models loaded on {device}!")

Loading BLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ FLAN-T5 ready!

All models loaded on cuda!


Core Pipeline Functions

In [4]:
from frame_extractor import extract_frames
from sklearn.cluster import KMeans


def download_youtube_video(url, output_path="/tmp/yt_video.mp4"):
    """Download YouTube video using yt-dlp."""
    import subprocess
    cmd = [
        "yt-dlp",
        "-f", "mp4[height<=480]/best[height<=480]/best",
        "-o", output_path,
        "--no-playlist",
        "--max-filesize", "50m",
        url
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if os.path.exists(output_path):
        return output_path, None
    else:
        return None, result.stderr


def caption_frame(frame):
    """Generate BLIP caption for one frame."""
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(frame_rgb)
    inputs = blip_processor(image, return_tensors="pt").to(device)
    with torch.no_grad():
        output = blip_model.generate(**inputs, max_new_tokens=50)
    return blip_processor.decode(output[0], skip_special_tokens=True)


def t5_fusion(captions):
    """Fuse multiple captions using FLAN-T5."""
    if len(captions) == 0:
        return ""
    if len(captions) == 1:
        return captions[0]
    unique = list(dict.fromkeys(captions))
    prompt = (
        f"These are descriptions of different frames from the same video: "
        f"{'. '.join(unique)}. "
        f"Write one concise sentence describing what is happening in the video:"
    )
    inputs = t5_tokenizer(prompt, return_tensors="pt",
                          max_length=512, truncation=True).to(device)
    with torch.no_grad():
        outputs = t5_model.generate(**inputs, max_new_tokens=50,
                                    num_beams=4, early_stopping=True)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)


def get_clip_embeddings(frames):
    """Get CLIP embeddings for a list of frames."""
    pil_images = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
                  for f in frames]
    inputs = clip_processor(images=pil_images,
                            return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        embeddings = clip_model.visual_projection(outputs.pooler_output)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
    return embeddings.cpu().numpy()


def strategy_a(frames, K=8):
    """Uniform sampling."""
    if K >= len(frames):
        indices = list(range(len(frames)))
    else:
        indices = [int(i * (len(frames)-1) / (K-1)) for i in range(K)]
    keyframes = [frames[i] for i in indices]
    caption = caption_frame(keyframes[len(keyframes)//2])
    return keyframes, indices, caption


def strategy_b(frames, threshold=0.7, max_kf=8):
    """SSIM scene change detection."""
    from skimage.metrics import structural_similarity as ssim
    keyframe_indices = [0]
    for i in range(1, len(frames)):
        g1 = cv2.cvtColor(cv2.resize(frames[i-1], (160,120)),
                          cv2.COLOR_BGR2GRAY)
        g2 = cv2.cvtColor(cv2.resize(frames[i], (160,120)),
                          cv2.COLOR_BGR2GRAY)
        score, _ = ssim(g1, g2, full=True)
        if score < threshold:
            keyframe_indices.append(i)
    if len(keyframe_indices) > max_kf:
        keyframe_indices = sorted(keyframe_indices)[:max_kf]
    if len(keyframe_indices) < 2:
        step = len(frames) // max_kf
        keyframe_indices = list(range(0, len(frames), max(step,1)))[:max_kf]
    keyframes = [frames[i] for i in keyframe_indices]
    caption = caption_frame(keyframes[len(keyframes)//2])
    return keyframes, keyframe_indices, caption


def strategy_c(frames, K=8):
    """CLIP K-Means clustering."""
    embeddings = get_clip_embeddings(frames)
    K = min(K, len(frames))
    kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
    kmeans.fit(embeddings)
    indices = []
    for k in range(K):
        mask = kmeans.labels_ == k
        cluster_idx = np.where(mask)[0]
        if len(cluster_idx) == 0:
            continue
        center = kmeans.cluster_centers_[k]
        dists = np.linalg.norm(embeddings[cluster_idx] - center, axis=1)
        indices.append(int(cluster_idx[np.argmin(dists)]))
    indices = sorted(indices)
    keyframes = [frames[i] for i in indices]
    caption = caption_frame(keyframes[len(keyframes)//2])
    return keyframes, indices, caption


def strategy_aplus(frames, K=8):
    """Uniform sampling + T5 fusion."""
    keyframes, indices, _ = strategy_a(frames, K)
    captions = [caption_frame(f) for f in keyframes]
    caption = t5_fusion(captions)
    return keyframes, indices, caption, captions


def strategy_bplus(frames, threshold=0.7, max_kf=8):
    """SSIM + T5 fusion."""
    keyframes, indices, _ = strategy_b(frames, threshold, max_kf)
    captions = [caption_frame(f) for f in keyframes]
    caption = t5_fusion(captions)
    return keyframes, indices, caption, captions


def strategy_cplus(frames, K=8, sim_threshold=0.85):
    """CLIP K-Means + redundancy filtering + T5 fusion."""
    keyframes, indices, _ = strategy_c(frames, K)
    if len(keyframes) > 1:
        embeddings = get_clip_embeddings(keyframes)
        kept = [0]
        for i in range(1, len(keyframes)):
            redundant = any(
                float(np.dot(embeddings[i], embeddings[j])) > sim_threshold
                for j in kept
            )
            if not redundant:
                kept.append(i)
        keyframes = [keyframes[i] for i in kept]
        indices = [indices[i] for i in kept]
    captions = [caption_frame(f) for f in keyframes]
    caption = t5_fusion(captions)
    return keyframes, indices, caption, captions


print("✅ All pipeline functions ready!")

✅ All pipeline functions ready!


Main Caption Function

In [5]:
def generate_caption(video_path, strategy_name):
    """
    Main function that runs the full pipeline on a video.
    Returns caption, keyframe images, and info string.
    """
    start = time.time()

    # Extract frames
    frames, fps, total = extract_frames(video_path)

    if len(frames) == 0:
        return "Error: Could not extract frames", [], "Failed"

    # Run selected strategy
    frame_captions = None

    if strategy_name == "A — Uniform Sampling":
        keyframes, indices, caption = strategy_a(frames)

    elif strategy_name == "B — SSIM Scene Detection":
        keyframes, indices, caption = strategy_b(frames)

    elif strategy_name == "C — CLIP K-Means":
        keyframes, indices, caption = strategy_c(frames)

    elif strategy_name == "A+ — Uniform + T5 Fusion":
        keyframes, indices, caption, frame_captions = strategy_aplus(frames)

    elif strategy_name == "B+ — SSIM + T5 Fusion":
        keyframes, indices, caption, frame_captions = strategy_bplus(frames)

    elif strategy_name == "C+ — CLIP + T5 Fusion":
        keyframes, indices, caption, frame_captions = strategy_cplus(frames)

    else:
        return "Unknown strategy", [], "Failed"

    elapsed = time.time() - start

    # Convert keyframes to PIL for Gradio display
    pil_frames = [
        Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        for f in keyframes
    ]

    # Build info string
    info = (
        f"Strategy: {strategy_name}\n"
        f"Video: {total} frames at {fps:.1f} FPS "
        f"({total/fps:.1f} seconds)\n"
        f"Keyframes selected: {len(keyframes)} "
        f"(indices: {indices})\n"
        f"Time taken: {elapsed:.2f} seconds"
    )

    if frame_captions:
        info += f"\n\nIndividual frame captions:\n"
        for i, fc in enumerate(frame_captions):
            info += f"  Frame {i+1}: {fc}\n"

    return caption, pil_frames, info

Gradio Interface

In [ ]:
def process_input(youtube_url, video_file, strategy):
    """Handle both URL and file upload inputs."""

    video_path = None
    error_msg = None

    # Determine input source
    if youtube_url and youtube_url.strip():
        # Download from YouTube
        tmp_path = "/tmp/yt_video.mp4"
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        video_path, error_msg = download_youtube_video(
            youtube_url.strip(), tmp_path)
        if error_msg:
            return (f"❌ Download failed: {error_msg}",
                    [], "Download error")

    elif video_file is not None:
        video_path = video_file

    else:
        return ("❌ Please provide a YouTube URL or upload a video file.",
                [], "No input provided")

    # Generate caption
    try:
        caption, keyframes, info = generate_caption(video_path, strategy)
        return caption, keyframes, info
    except Exception as e:
        return f"❌ Error: {str(e)}", [], "Pipeline error"


# Build Gradio UI
with gr.Blocks(title="Video Captioning — IIT Kharagpur") as demo:

    gr.Markdown("""
    # 🎬 Video Captioning — IIT Kharagpur Thesis
    ### Keyframe Selection Strategies for Automated Video Captioning
    *M.Tech Thesis — Department of Mathematics, IIT Kharagpur*
    ---
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input")

            youtube_url = gr.Textbox(
                label="YouTube URL (Shorts or regular)",
                placeholder="https://www.youtube.com/shorts/...",
                lines=1
            )

            gr.Markdown("**OR**")

            video_file = gr.Video(
                label="Upload Video File",
                sources=["upload"]
            )

            strategy = gr.Dropdown(
                choices=[
                    "A — Uniform Sampling",
                    "B — SSIM Scene Detection",
                    "C — CLIP K-Means",
                    "A+ — Uniform + T5 Fusion",
                    "B+ — SSIM + T5 Fusion",
                    "C+ — CLIP + T5 Fusion",
                ],
                value="C+ — CLIP + T5 Fusion",
                label="Select Strategy"
            )

            submit_btn = gr.Button(
                "▶ Generate Caption",
                variant="primary",
                size="lg"
            )

        with gr.Column(scale=1):
            gr.Markdown("### Output")

            caption_out = gr.Textbox(
                label="Generated Caption",
                lines=3,
                interactive=False
            )

            keyframes_out = gr.Gallery(
                label="Selected Keyframes",
                columns=4,
                rows=2,
                height=300
            )

            info_out = gr.Textbox(
                label="Processing Details",
                lines=8,
                interactive=False
            )

    submit_btn.click(
        fn=process_input,
        inputs=[youtube_url, video_file, strategy],
        outputs=[caption_out, keyframes_out, info_out]
    )

    gr.Markdown("""
    ---
    **Strategies explained:**
    - **A** — Selects 8 evenly spaced frames, captions the middle frame
    - **B** — Detects scene changes using SSIM, captions representative frame
    - **C** — Uses CLIP embeddings + K-Means to select diverse frames
    - **A+/B+/C+** — Same selection + captions ALL frames + T5 fusion
    """)

# Launch in Colab
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f250c9fd6ab49f8617.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Test With a YouTube URL

In [ ]:
# Test the pipeline directly first
test_url = "https://www.youtube.com/shorts/PASTE_ANY_SHORTS_URL_HERE"

# Test download
video_path, error = download_youtube_video(test_url)
if video_path:
    print(f"✅ Downloaded: {video_path}")
    caption, keyframes, info = generate_caption(
        video_path, "C+ — CLIP + T5 Fusion")
    print(f"Caption: {caption}")
    print(f"Info: {info}")
else:
    print(f"❌ Error: {error}")